<a href="https://colab.research.google.com/github/moucheng2017/my_simsiam/blob/main/notebooks/topk-categorical-bottleneck-pic-cifar10-sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Top-k Categorical Bottleneck PIC Sweep

This notebook trains a PIC-style instance classifier through a sampled top-k categorical bottleneck. The experiment asks whether constraining instance prediction through `C` latent categories with `k` active categories changes the kNN-minus-linear-probe gap. Every run trains for 100 epochs and then runs linear evaluation.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is only available inside Google Colab; skipping Drive mount.')

In [ ]:
import getpass
import os
import subprocess
from pathlib import Path
from urllib.parse import quote

PUBLIC_REPO_URL = 'https://github.com/moucheng2017/instance_self_sup.git'
BRANCH = '1-vmf-pseudo-sup'  # Change to your experiment branch before running in Colab.
REPO_DIR = Path('/content/instance_self_sup')
GITHUB_TOKEN = ''

def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass

repo_url = PUBLIC_REPO_URL
if GITHUB_TOKEN:
    repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(GITHUB_TOKEN, safe="")}@')

def prompt_token():
    raw = getpass.getpass('GitHub token (leave blank to stop): ')
    return raw.strip() if isinstance(raw, str) else ''

os.chdir('/content')
if not REPO_DIR.exists():
    try:
        run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
    except subprocess.CalledProcessError as exc:
        if GITHUB_TOKEN:
            raise
        token = prompt_token()
        if not token:
            raise RuntimeError('Clone failed and no token was provided.') from exc
        repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(token, safe="")}@')
        run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
else:
    print(f'Repo already exists at {REPO_DIR}; fetching {BRANCH}.')
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
run(['pip', 'install', '-r', 'requirements_colab.txt'], cwd=REPO_DIR)

In [ ]:
import os
import torchvision

DATA_DIR = '/content/drive/MyDrive/SSL_instance_sup/data'
os.makedirs(DATA_DIR, exist_ok=True)
torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)
print('CIFAR-10 is ready.')

In [ ]:
from colab_utils import linear_eval_from_colab, train_from_colab

CONFIG = 'configs/topk_categorical_bottleneck_pic_cifar_colab.yaml'

# Capacity sweep. Start small; expand if the trend is informative.
SWEEP = [
    {'C': 50, 'k': 1},
    {'C': 100, 'k': 5},
    {'C': 500, 'k': 5},
    {'C': 1000, 'k': 5},
    {'C': 5000, 'k': 5},
]
EPOCHS = 100
BATCH_SIZE = 1024
SAMPLES_PER_EPOCH = 20480
SOURCE_POOL_SIZE = 50000
BALANCE_WEIGHT = 1.0
ENTROPY_WEIGHT = 0.0
COLUMN_NORMALIZE = True

results = []

In [ ]:
for item in SWEEP:
    C, k = int(item['C']), int(item['k'])
    run_name = f'topk-cat-bottleneck-pic-cifar10-C{C}-k{k}'
    overrides = {
        'name': run_name,
        'model': {
            'num_latent_classes': C,
            'topk': k,
            'balance_weight': BALANCE_WEIGHT,
            'entropy_weight': ENTROPY_WEIGHT,
            'column_normalize': COLUMN_NORMALIZE,
            'decoder_hidden_dim': None,
        },
        'train': {
            'num_epochs': EPOCHS,
            'stop_at_epoch': EPOCHS,
            'batch_size': BATCH_SIZE,
            'source_pool_size': SOURCE_POOL_SIZE,
            'samples_per_epoch': SAMPLES_PER_EPOCH,
        },
    }
    train_result = train_from_colab(
        CONFIG,
        project_name='SSL_exps_instance_sup',
        use_drive=True,
        overrides=overrides,
        device='cuda',
        debug=False,
        download=False,
        hide_progress=False,
    )
    linear_result = linear_eval_from_colab(
        CONFIG,
        checkpoint_path=train_result['model_path'],
        project_name='SSL_exps_instance_sup',
        use_drive=True,
        overrides={
            'name': f'{run_name}-linear-eval',
            'model': {'num_latent_classes': C, 'topk': k},
            'eval': {
                'optimizer': {'name': 'sgd', 'weight_decay': 0.0, 'momentum': 0.9},
                'warmup_epochs': 0,
                'warmup_lr': 0,
                'base_lr': 0.1,
                'final_lr': 0,
                'num_epochs': 100,
                'batch_size': 1024,
            },
        },
        device='cuda',
        debug=False,
        download=False,
        hide_progress=False,
    )
    row = {
        'num_latent_classes': C,
        'topk': k,
        'knn_accuracy': train_result['accuracy'],
        'linear_accuracy': linear_result['accuracy'],
        'gap_knn_minus_linear': train_result['accuracy'] - linear_result['accuracy'],
        'checkpoint': train_result['model_path'],
        'log_dir': train_result['log_dir'],
    }
    results.append(row)
    print(row)

results